# 03 — Physics-Aware Feature Engineering

**FusionCore v0 — Phase 3**

This notebook constructs the **Dynamic Feature Vector** from the unified,
regime-normalised dataset `FD00u` produced in Phase 2. It has two responsibilities:

1. **Gap Resolution (Part A):** Address every outstanding gap identified in the
   Phase 1 and Phase 2 technical reports before the feature matrix is passed to
   Phase 4.
2. **Feature Construction (Part B):** Engineer the physics-informed feature space
   following the four-step physics-informed pipeline.

### Gap Resolution Mandate

| # | Gap | Source | Severity | Status |
|---|-----|--------|----------|--------|
| G1 | Unit_id global uniqueness | P2 §9.2 | MANDATORY | Resolved |
| G2 | Row temporal ordering | P2 §9.2 | MANDATORY | Resolved |
| G3 | Dataset class imbalance | P1 §7.2, P2 §9.2 | DIAGNOSTIC | Documented; deferred to Phase 4 |
| G4 | Regime row imbalance | P1 §7.2, P2 §9.2 | DIAGNOSTIC | Documented; deferred to Phase 4 |
| G5 | regime_id integer encoding | P2 §9.2 | MANDATORY | Resolved |
| G6 | s7 cross-fault-mode sigma differential (3.93x) | P2 §9.2 | MANDATORY | Resolved |
| G7 | Mechanically linked pairs (s8/s13, s9/s14) | P1 §7.2, P2 §9.2 | MANDATORY | Resolved |
| G8 | Extreme z-score excursions | P1 §7.2, P2 §9.2 | MANDATORY | Resolved |
| G9 | s16 (farB) partial-dead | P2 §9.2 | DIAGNOSTIC | Documented; retained |
| G10 | Multivariate health index | P2 §9.2 | MANDATORY | Resolved |
| G11 | Regime assignment OOD guard | P2 §9.2 | DIAGNOSTIC | Documented; deferred to v1 |
| G12 | Wide P-F spread: survival analysis framework | P2 §6.3.3 | DIAGNOSTIC | Documented; deferred to Phase 5 |
| G13 | Wide P-F spread: per-fault-mode-family UWL recalibration | P2 §6.3.3 | DIAGNOSTIC | Documented; deferred to Phase 5 |

### Feature Engineering Pipeline

| Step | Description | Features Added | Running Total |
|------|-------------|----------------|---------------|
| Step 1 | Core Manifold (3 ops + 21 sensors) | 24 | 24 |
| Step 2 | Kinematic Expansion (delta, rolling mean, rolling sigma) | N_active x 3 | Derived at runtime |
| Step 3 | Thermodynamic Virtual Sensors (CPR, E_thermal, EGT Drift) | 3 | Derived at runtime |
| Step 4 | Cumulative Fatigue Index (Miner's Rule proxy) | 3 | **Derived at runtime** |

**References:**
- Saxena, A. & Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data Set.* NASA Ames.
- Heimes, F.O. (2008). Recurrent neural networks for RUL estimation. *PHM 2008.*
- Paris, P.C. & Erdogan, F. (1963). A critical analysis of crack propagation laws.

In [ ]:
# ============================================================================
# Cell 1 — Environment Setup (Run First)
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/PI/Datasets')

# G12 survival analysis dependency.
!pip install -q lifelines

## Dependencies

All libraries used in this notebook (NumPy, Pandas, Matplotlib, Seaborn,
Scikit-learn, Joblib, SciPy) are pre-installed in Google Colab. No additional
`pip install` commands are required.

> **Additional dependency:** `lifelines` (survival analysis — Kaplan-Meier, Cox PH).

In [ ]:
# ============================================================================
# Cell 3 — Project Constants & Data Loader
# ============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

CMAPSS_DIR      = Path('/content/drive/MyDrive/PI/Datasets/CMAPSS')
OUTPUTS_DIR     = Path('/content/drive/MyDrive/PI/FusionCore/v0/outputs')
REGIME_DICT_DIR = OUTPUTS_DIR / 'regime_dictionary'

CMAPSS_SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
CMAPSS_COLUMNS = [
    "unit_id", "cycle",
    "op1", "op2", "op3",
    "s1",  "s2",  "s3",  "s4",  "s5",  "s6",  "s7",
    "s8",  "s9",  "s10", "s11", "s12", "s13", "s14",
    "s15", "s16", "s17", "s18", "s19", "s20", "s21",
]

SENSOR_COLS = [c for c in CMAPSS_COLUMNS if c.startswith("s")]
OP_COLS     = ["op1", "op2", "op3"]

RUL_CAP            = 125       # Heimes (2008)
VARIANCE_THRESHOLD = 1.0e-5    # tau — dead-sensor detection threshold
RANDOM_STATE       = 42
ROLLING_WINDOW     = 5         # Kinematic expansion window (section 13.3)

# Virtual sensor column mappings (section 8.3).
CPR_NUMERATOR   = "s7"         # P30 — HPC outlet total pressure
CPR_DENOMINATOR = "s3"         # T30 — HPC outlet temperature
# NOTE: Original spec (§13) used s7/s5 (P30/P2).  s5 is regime-dead
# in z-space (var ≈ 0), making that ratio degenerate.  Replaced with
# s7/s3 (P30/T30) — total-pressure-to-temperature at HPC outlet —
# proportional to gas density, a meaningful compressor-health proxy.
E_THERMAL_NUM_A = "s12"        # phi — Fuel/Ps30 ratio
E_THERMAL_NUM_B = "s11"        # Ps30 — HPC static pressure
E_THERMAL_DENOM = "s9"         # Nc  — Core speed
EGT_SENSOR      = "s4"         # T50 — LPT outlet / EGT

# Cumulative fatigue sensors (section 13.5 — Miner's Rule).
FATIGUE_SENSORS = ["s4", "s9", "s7"]  # T50 (EGT), Nc (Core Speed), P30

# Mechanically linked pairs (G7).
LINKED_PAIRS = [("s8", "s13"), ("s9", "s14")]  # (Nf, NRf), (Nc, NRc)

# Sensor physical names for display.
SENSOR_NAMES = {
    "s1": "T2 (Fan Inlet Temp)",        "s2": "T24 (LPC Outlet Temp)",
    "s3": "T30 (HPC Outlet Temp)",      "s4": "T50 (LPT Outlet / EGT)",
    "s5": "P2 (Fan Inlet Pressure)",    "s6": "P15 (Bypass Duct Pressure)",
    "s7": "P30 (HPC Outlet Pressure)",  "s8": "Nf (Fan Speed)",
    "s9": "Nc (Core Speed)",            "s10": "epr (Engine Pressure Ratio)",
    "s11": "Ps30 (HPC Static Pres.)",   "s12": "phi (Fuel/Ps30 Ratio)",
    "s13": "NRf (Corrected Fan Speed)", "s14": "NRc (Corrected Core Speed)",
    "s15": "BPR (Bypass Ratio)",        "s16": "farB (Burner Fuel-Air Ratio)",
    "s17": "htBleed (Bleed Enthalpy)",  "s18": "Nf_dmd (Demanded Fan Speed)",
    "s19": "PCNfR_dmd (Dem. Corr. Fan)","s20": "W31 (HPT Coolant Bleed)",
    "s21": "W32 (LPT Coolant Bleed)",
}

print("Project constants loaded.")
print(f"  OUTPUTS_DIR:     {OUTPUTS_DIR}")
print(f"  Sensors:         {len(SENSOR_COLS)}")
print(f"  Rolling window:  {ROLLING_WINDOW}")
print(f"  Fatigue sensors: {FATIGUE_SENSORS}")

In [ ]:
# ============================================================================
# Cell 4 — Imports & Visualisation Setup
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import joblib
import os
from scipy import stats

# FusionCore colour palette.
FC_DARK_BLUE  = '#0D1B2A'
FC_NAVY       = '#1B3A5C'
FC_ORANGE     = '#D96A1B'
FC_DEEP_RED   = '#9B1B30'
FC_STEEL      = '#4A6274'
FC_CHARCOAL   = '#2D2D2D'
FC_LIGHT_GREY = '#E8E8E8'

FC_PALETTE = [FC_DARK_BLUE, FC_ORANGE, FC_DEEP_RED, FC_STEEL, FC_NAVY]

plt.rcParams.update({
    'figure.figsize':       (14, 5),
    'figure.dpi':           150,
    'savefig.dpi':          300,
    'savefig.bbox':         'tight',
    'font.family':          'serif',
    'font.size':            11,
    'axes.titlesize':       13,
    'axes.titleweight':     'bold',
    'axes.labelsize':       11,
    'axes.edgecolor':       FC_CHARCOAL,
    'axes.labelcolor':      FC_CHARCOAL,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'axes.prop_cycle':      mpl.cycler(color=FC_PALETTE),
    'xtick.color':          FC_CHARCOAL,
    'ytick.color':          FC_CHARCOAL,
    'legend.fontsize':      9,
    'legend.framealpha':    0.9,
    'grid.color':           FC_LIGHT_GREY,
    'grid.alpha':           0.6,
    'grid.linestyle':       ':',
})

print("Imports and visualisation configuration complete.")

In [ ]:
# ============================================================================
# Cell 5 — Load FD00u from Phase 2 Outputs
# ============================================================================

fd00u_train = pd.read_parquet(OUTPUTS_DIR / "fd00u_train.parquet")
fd00u_val   = pd.read_parquet(OUTPUTS_DIR / "fd00u_val.parquet")

print(f"FD00u (Train):      {fd00u_train.shape[0]:>7,} rows, "
      f"{fd00u_train.groupby('subset_origin')['unit_id'].nunique().sum()} engines")
print(f"FD00u (Validation): {fd00u_val.shape[0]:>7,} rows, "
      f"{fd00u_val.groupby('subset_origin')['unit_id'].nunique().sum()} engines")
print(f"\nColumns ({len(fd00u_train.columns)}): {list(fd00u_train.columns)}")
print(f"\nSubset distribution (Train):")
print(fd00u_train['subset_origin'].value_counts().sort_index())

---

## Part A — Gap Resolution

All gaps identified in the Phase 1 and Phase 2 technical reports are addressed
below. MANDATORY gaps are resolved with code changes that modify the downstream
feature matrix. DIAGNOSTIC gaps are analysed, documented, and a disposition
recorded.

In [ ]:
# ============================================================================
# Cell 7 — G1: Composite Key Enforcement & G2: Temporal Ordering
# ============================================================================

# G1: unit_id is NOT globally unique across subsets. The composite key
# (subset_origin, unit_id) uniquely identifies each engine trajectory.
# All downstream groupby operations must use this composite key.

UNIT_KEY = ["subset_origin", "unit_id"]

def enforce_composite_key(df, name):
    """
    G1 + G2: Enforce composite key uniqueness and temporal ordering.
    Sort by (subset_origin, unit_id, cycle) to guarantee monotonic time
    within each engine trajectory — prerequisite for rolling/diff operations.
    """
    df = df.sort_values(UNIT_KEY + ["cycle"]).reset_index(drop=True)

    # Verify no duplicate (subset_origin, unit_id, cycle) tuples.
    dupes = df.duplicated(subset=UNIT_KEY + ["cycle"], keep=False).sum()
    assert dupes == 0, (
        f"[{name}] Found {dupes} duplicate (subset_origin, unit_id, cycle) rows."
    )

    # Verify monotonic cycle within each engine.
    mono_check = df.groupby(UNIT_KEY)["cycle"].apply(lambda x: x.is_monotonic_increasing)
    assert mono_check.all(), (
        f"[{name}] Cycle is not monotonically increasing for all engines."
    )

    print(f"  [{name}] Composite key enforced — {df.shape[0]:,} rows, "
          f"{df.groupby('subset_origin')['unit_id'].nunique().sum()} engines, "
          f"0 duplicates, all trajectories monotonically ordered.")
    return df

print("G1 — Composite Key Enforcement")
print("G2 — Temporal Ordering\n")

fd00u_train = enforce_composite_key(fd00u_train, "Train")
fd00u_val   = enforce_composite_key(fd00u_val, "Val")

print("\nG1 and G2 resolved.")

In [ ]:
# ============================================================================
# Cell 8 — G3: Dataset Class Imbalance (DIAGNOSTIC)
# ============================================================================

print("G3 — Dataset Class Imbalance Diagnostic\n")

# Row count distribution by subset.
subset_counts = fd00u_train.groupby("subset_origin").size()
subset_pct = (subset_counts / subset_counts.sum() * 100).round(1)

print("Subset row distribution (Internal Train):")
for s in CMAPSS_SUBSETS:
    if s in subset_counts.index:
        print(f"  {s}: {subset_counts[s]:>6,} rows ({subset_pct[s]}%)")

# Terminal cycle distribution (RUL < 30).
terminal = fd00u_train[fd00u_train["RUL"] < 30]
terminal_pct = len(terminal) / len(fd00u_train) * 100
print(f"\nTerminal rows (RUL < 30): {len(terminal):,} ({terminal_pct:.1f}%)")

# Assign per-subset colours from the FusionCore palette.
SUBSET_COLOURS = {s: FC_PALETTE[i] for i, s in enumerate(CMAPSS_SUBSETS)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Row count per subset (bar chart).
bar_colours = [SUBSET_COLOURS[s] for s in subset_counts.index]
subset_counts.plot(kind="bar", ax=axes[0], color=bar_colours, edgecolor=FC_CHARCOAL)
axes[0].set_title("Row Count by Subset (Internal Train)")
axes[0].set_ylabel("Row Count")
axes[0].set_xlabel("Subset")
axes[0].tick_params(axis='x', rotation=0)

# Right panel: Per-subset KDE of RUL (matching colours, no overlap confusion).
for s in CMAPSS_SUBSETS:
    subset_data = fd00u_train[fd00u_train["subset_origin"] == s]["RUL"]
    if len(subset_data) > 0:
        sns.kdeplot(subset_data, ax=axes[1], color=SUBSET_COLOURS[s],
                    linewidth=2, label=s, fill=True, alpha=0.15)
axes[1].set_title("RUL Density by Subset (Internal Train)")
axes[1].set_xlabel("RUL (cycles)")
axes[1].set_ylabel("Density")
axes[1].legend(title="Subset")

plt.tight_layout()
plt.show()

print("\nG3 documented.")
print("  Disposition: XGBoost (Phase 4) is robust to class imbalance via gradient")
print("  boosting. Subset-stratified sampling evaluated in Phase 4 if needed.")

In [ ]:
# ============================================================================
# Cell 9 — G4: Regime Row Imbalance (DIAGNOSTIC)
# ============================================================================

print("G4 — Regime Row Imbalance Diagnostic\n")

if "regime_id" in fd00u_train.columns:
    regime_counts = fd00u_train.groupby(["subset_origin", "regime_id"]).size().unstack(fill_value=0)
    print("Regime distribution by subset (Internal Train):")
    print(regime_counts)
    print()

    overall_regime = fd00u_train["regime_id"].value_counts().sort_index()
    overall_pct = (overall_regime / overall_regime.sum() * 100).round(1)
    print("Overall regime distribution:")
    for r in overall_regime.index:
        print(f"  Regime {r}: {overall_regime[r]:>6,} rows ({overall_pct[r]}%)")

    fig, ax = plt.subplots(figsize=(10, 5))
    regime_counts.T.plot(kind="bar", ax=ax, color=FC_PALETTE[:4], edgecolor=FC_CHARCOAL)
    ax.set_title("Regime Row Distribution by Subset")
    ax.set_xlabel("Regime ID")
    ax.set_ylabel("Row Count")
    ax.legend(title="Subset")
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("  regime_id column not present in FD00u.")
    print("  Regime information was used for normalisation in Phase 2 but the")
    print("  integer regime_id may not have been saved in the parquet output.")

print("\nG4 documented.")
print("  Disposition: Inverse-frequency weighting may be relevant for gradient-based")
print("  models in Phase 4. Not required for XGBoost in Phase 4.")

In [ ]:
# ============================================================================
# Cell 10 — G5: regime_id Encoding (MANDATORY)
# ============================================================================

print("G5 — regime_id Encoding Resolution\n")

# Integer regime_id implies ordinal relationship that does not exist.
# Resolution: One-hot encode regime_id. The raw operational settings (op1, op2, op3)
# are already retained as continuous features in the core manifold (Step 1).

def encode_regime_id(df, name):
    """One-hot encode regime_id and drop the integer column."""
    if "regime_id" not in df.columns:
        print(f"  [{name}] regime_id not in columns — skipping one-hot encoding.")
        print(f"           Raw operational settings (op1, op2, op3) are retained.")
        return df

    dummies = pd.get_dummies(df["regime_id"], prefix="regime", dtype=int)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=["regime_id"], inplace=True)

    regime_cols = [c for c in df.columns if c.startswith("regime_")]
    print(f"  [{name}] regime_id one-hot encoded -> {regime_cols}")
    return df

fd00u_train = encode_regime_id(fd00u_train, "Train")
fd00u_val   = encode_regime_id(fd00u_val, "Val")

print("\nG5 resolved — integer regime_id replaced with one-hot columns.")

In [ ]:
# ============================================================================
# Cell 11 — G6: s7 Cross-Fault-Mode Sigma Differential (MANDATORY)
# ============================================================================

print("G6 — s7 Cross-Fault-Mode Sigma Differential Resolution\n")

# Within-regime sigma for s7 differs 3.93x between single-fault (FD001/FD002)
# and dual-fault (FD003/FD004).
# Resolution: Add binary fault_mode_family indicator feature.

SINGLE_FAULT_SUBSETS = ["FD001", "FD002"]
DUAL_FAULT_SUBSETS   = ["FD003", "FD004"]

def add_fault_mode_family(df, name):
    """Add binary fault_mode_family indicator."""
    df["fault_mode_family"] = df["subset_origin"].apply(
        lambda x: 0 if x in SINGLE_FAULT_SUBSETS else 1
    )
    counts = df["fault_mode_family"].value_counts().sort_index()
    print(f"  [{name}] fault_mode_family distribution:")
    print(f"    0 (single-fault): {counts.get(0, 0):>6,} rows")
    print(f"    1 (dual-fault):   {counts.get(1, 0):>6,} rows")
    return df

fd00u_train = add_fault_mode_family(fd00u_train, "Train")
fd00u_val   = add_fault_mode_family(fd00u_val, "Val")

# Verify the sigma differential for documentation.
s7_single = fd00u_train[fd00u_train["fault_mode_family"] == 0]["s7"]
s7_dual   = fd00u_train[fd00u_train["fault_mode_family"] == 1]["s7"]
print(f"\n  s7 sigma (single-fault): {s7_single.std():.4f}")
print(f"  s7 sigma (dual-fault):   {s7_dual.std():.4f}")
if s7_single.std() > 0:
    print(f"  Ratio: {s7_dual.std() / s7_single.std():.2f}x")

print("\nG6 resolved — fault_mode_family indicator added.")

In [ ]:
# ============================================================================
# Cell 12 — G7: Mechanically Linked Sensor Pairs — VIF Analysis (MANDATORY)
# ============================================================================

print("G7 — Mechanically Linked Sensor Pairs (s8/s13, s9/s14)\n")

# s8 (Nf) and s13 (NRf) are physical vs corrected fan speed.
# s9 (Nc) and s14 (NRc) are physical vs corrected core speed.
# NRx = Nx / sqrt(theta) — structural multicollinearity.
#
# Resolution: Compute VIF. Construct ratio features s8/s13 and s9/s14 which
# capture the correction factor (theta). Both raw sensors retained per
# Critical Prohibition #3 (no sensor removal).

print("Variance Inflation Factor (VIF) analysis on linked pairs:\n")

for s_phys, s_corr in LINKED_PAIRS:
    phys_name = SENSOR_NAMES.get(s_phys, s_phys)
    corr_name = SENSOR_NAMES.get(s_corr, s_corr)
    corr_val = fd00u_train[s_phys].corr(fd00u_train[s_corr])
    vif = 1 / (1 - corr_val**2) if abs(corr_val) < 1.0 else float("inf")
    print(f"  {s_phys} ({phys_name}) vs {s_corr} ({corr_name}):")
    print(f"    Pearson r = {corr_val:.4f}, VIF = {vif:.1f}")

# Construct derived ratio features.
# Safe division: in z-normalised space, denominators can be near-zero.
# Guard threshold matches the virtual-sensor threshold in Cell 22.
SAFE_DENOM_THRESHOLD_G7 = 1e-6

print("\nConstructing ratio features (safe division, |denom| < 1e-6 -> 0):")

for s_phys, s_corr in LINKED_PAIRS:
    ratio_col = f"{s_phys}_{s_corr}_ratio"

    for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
        denom = df_loop[s_corr]
        safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD_G7, np.nan)
        df_loop[ratio_col] = (df_loop[s_phys] / safe_denom).fillna(0)

    print(f"  {ratio_col} — mean: {fd00u_train[ratio_col].mean():.4f}, "
          f"std: {fd00u_train[ratio_col].std():.4f}")

print("\nG7 resolved — raw sensors retained; ratio features added.")
print("  Regularisation in Phase 4/5 will manage residual multicollinearity.")

In [ ]:
# ============================================================================
# Cell 13 — G8: Extreme Z-Score Excursions — Cycle-Position Analysis (MANDATORY)
# ============================================================================

print("G8 — Extreme Z-Score Excursion Analysis\n")

# Sensors with extreme z-scores: s6 (-7.9), s8 (8.3), s9 (8.9), s13 (7.5), s14 (8.9).
# Do extremes cluster at initialisation (transient artefacts -> Winsorise)
# or at terminal cycles (genuine degradation -> retain)?

EXCURSION_SENSORS = ["s6", "s8", "s9", "s13", "s14"]
EXCURSION_THRESHOLD = 5.0

# Compute normalised cycle position (0 = start, 1 = end of life).
for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
    max_cyc = df_loop.groupby(UNIT_KEY)["cycle"].transform("max")
    df_loop["cycle_position"] = df_loop["cycle"] / max_cyc

print("Cycle-position distribution of extreme z-scores (|z| > 5.0):\n")

excursion_results = {}
fig, axes = plt.subplots(1, len(EXCURSION_SENSORS), figsize=(18, 4), sharey=True)

for i, sensor in enumerate(EXCURSION_SENSORS):
    mask = fd00u_train[sensor].abs() > EXCURSION_THRESHOLD
    extreme_rows = fd00u_train[mask]
    n_extreme = len(extreme_rows)
    pct_extreme = n_extreme / len(fd00u_train) * 100

    if n_extreme > 0:
        pos = extreme_rows["cycle_position"]
        early = (pos < 0.2).sum()
        terminal_count = (pos > 0.8).sum()
        middle = n_extreme - early - terminal_count

        excursion_results[sensor] = {
            "total": n_extreme,
            "early_pct": early / n_extreme * 100,
            "terminal_pct": terminal_count / n_extreme * 100,
        }

        print(f"  {sensor} ({SENSOR_NAMES.get(sensor, '')}):")
        print(f"    Extreme rows: {n_extreme:,} ({pct_extreme:.2f}%)")
        print(f"    Early (<20% life): {early} ({early/n_extreme*100:.1f}%)")
        print(f"    Terminal (>80% life): {terminal_count} ({terminal_count/n_extreme*100:.1f}%)")
        print(f"    Middle: {middle} ({middle/n_extreme*100:.1f}%)")

        axes[i].hist(pos, bins=20, color=FC_DEEP_RED, edgecolor=FC_CHARCOAL, alpha=0.8)
        axes[i].set_title(f"{sensor} (n={n_extreme})")
        axes[i].set_xlabel("Cycle Position (0=start, 1=EOL)")
    else:
        print(f"  {sensor}: No extreme z-scores (|z| > {EXCURSION_THRESHOLD})")
        axes[i].set_title(f"{sensor} (n=0)")

axes[0].set_ylabel("Count")
fig.suptitle("Cycle Position of Extreme Z-Score Excursions (|z| > 5.0)", fontweight="bold")
plt.tight_layout()
plt.show()

# Disposition.
print("\n--- G8 Disposition ---")
terminal_dominant = 0
for sensor, res in excursion_results.items():
    if res["terminal_pct"] > res["early_pct"]:
        terminal_dominant += 1
        print(f"  {sensor}: Terminal-dominant -> RETAIN (genuine degradation)")
    else:
        print(f"  {sensor}: Early-dominant -> investigate transient artefacts")

print(f"\nOverall: {terminal_dominant}/{len(excursion_results)} sensors show "
      f"terminal-dominant excursions.")
print("Decision: RETAIN all extreme z-scores — they carry critical end-of-life signal.")
print("\nG8 resolved — no Winsorisation applied; excursions are degradation signals.")

# Remove temporary column.
fd00u_train.drop(columns=["cycle_position"], inplace=True)
fd00u_val.drop(columns=["cycle_position"], inplace=True)

In [ ]:
# ============================================================================
# Cell 14 — G9: s16 (farB) Partial-Dead Status (DIAGNOSTIC)
# ============================================================================

print("G9 — s16 (farB) Partial-Dead Status Diagnostic\n")

s16_stats = fd00u_train["s16"].describe()
s16_iqr = s16_stats["75%"] - s16_stats["25%"]

print(f"  s16 (farB — Burner Fuel-Air Ratio) statistics:")
print(f"    Mean:   {s16_stats['mean']:.6f}")
print(f"    Std:    {s16_stats['std']:.6f}")
print(f"    IQR:    {s16_iqr:.6f}")
print(f"    Min:    {s16_stats['min']:.6f}")
print(f"    Max:    {s16_stats['max']:.6f}")

print("\n  s16 variance by subset:")
for s in CMAPSS_SUBSETS:
    mask = fd00u_train["subset_origin"] == s
    if mask.any():
        var = fd00u_train.loc[mask, "s16"].var()
        print(f"    {s}: var = {var:.6f}")

print("\n  Disposition: s16 has near-zero IQR and minimal within-regime degradation")
print("  information. However, Critical Prohibition #3 prohibits sensor removal.")
print("  s16 is RETAINED. Ablation in Phase 4 will determine its contribution.")
print("\nG9 documented — s16 retained per pipeline constraint.")

In [ ]:
# ============================================================================
# Cell 15 — G10: Multivariate Health Index (MANDATORY)
# ============================================================================

print("G10 — Multivariate Health Index Construction\n")

# Univariate s4 alert is inadequate given the wide P-F spread (CV = 1.052).
# Construct composite health index from s4, s7, s9 — the three sensors confirmed
# to carry independent degradation signatures across all fault modes.

HEALTH_INDEX_SENSORS = ["s4", "s7", "s9"]

for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
    df_loop["health_index"] = df_loop[HEALTH_INDEX_SENSORS].mean(axis=1)

print(f"  Health Index = mean(s4, s7, s9)")
print(f"  Sensors: {[f'{s} ({SENSOR_NAMES[s]})' for s in HEALTH_INDEX_SENSORS]}")
print(f"\n  Train — health_index stats:")
print(f"    Mean: {fd00u_train['health_index'].mean():.4f}")
print(f"    Std:  {fd00u_train['health_index'].std():.4f}")
print(f"    Min:  {fd00u_train['health_index'].min():.4f}")
print(f"    Max:  {fd00u_train['health_index'].max():.4f}")

# Visualise for a sample engine.
sample_engine = fd00u_train.groupby(UNIT_KEY).size().idxmax()
sample_mask = (
    (fd00u_train["subset_origin"] == sample_engine[0]) &
    (fd00u_train["unit_id"] == sample_engine[1])
)
sample_df = fd00u_train[sample_mask].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sample_df["cycle"], sample_df["health_index"],
        color=FC_DEEP_RED, linewidth=1.5, label="Health Index")
ax.set_title(f"Multivariate Health Index — Engine ({sample_engine[0]}, "
             f"unit {sample_engine[1]})")
ax.set_xlabel("Cycle")
ax.set_ylabel("Health Index (z-space)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print("\nG10 resolved — health_index engineered from s4 + s7 + s9.")

In [ ]:
# ============================================================================
# Cell 16 — G11: Regime Assignment for Unseen Conditions (DIAGNOSTIC)
# ============================================================================

import warnings

print("G11 — Regime Assignment OOD Guard Diagnostic\n")

# K-Means has no out-of-distribution guard. Implement distance-to-nearest-
# centroid check to quantify regime assignment coverage.
# Phase 2 serialised individual K-Means models as kmeans_FD002.pkl and
# kmeans_FD004.pkl (not a single regime_dictionary.pkl).

kmeans_loaded = {}
for subset_key in ["FD002", "FD004"]:
    model_path = REGIME_DICT_DIR / f"kmeans_{subset_key}.pkl"
    if model_path.exists():
        kmeans_loaded[subset_key] = joblib.load(model_path)
        print(f"  {subset_key} K-Means model loaded from: {model_path}")
    else:
        print(f"  {subset_key} K-Means model not found at: {model_path}")

if kmeans_loaded:
    for subset_key, kmeans in kmeans_loaded.items():
        centroids = kmeans.cluster_centers_
        print(f"\n  {subset_key} K-Means centroids ({centroids.shape[0]} regimes):")
        for j, c in enumerate(centroids):
            print(f"    Regime {j}: op1={c[0]:.4f}, op2={c[1]:.4f}, op3={c[2]:.4f}")

        subset_mask = fd00u_train["subset_origin"] == subset_key
        if subset_mask.any():
            ops = fd00u_train.loc[subset_mask, OP_COLS].values
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore",
                    message="X does not have valid feature names",
                    category=UserWarning)
                dists = kmeans.transform(ops).min(axis=1)
            p95 = np.percentile(dists, 95)
            p99 = np.percentile(dists, 99)
            max_d = dists.max()
            print(f"    Distance-to-nearest-centroid: P95={p95:.4f}, "
                  f"P99={p99:.4f}, Max={max_d:.4f}")
            print(f"    Recommended OOD threshold: {p99:.4f} (P99)")
else:
    print("  No K-Means models found in regime dictionary directory.")
    print(f"  Searched: {REGIME_DICT_DIR}")
    contents = list(REGIME_DICT_DIR.iterdir()) if REGIME_DICT_DIR.exists() else []
    if contents:
        print(f"  Directory contents: {[f.name for f in contents]}")
    else:
        print("  Directory is empty or does not exist.")

print("\n  Disposition: Distance-to-nearest-centroid documented. Full OOD guard")
print("  implementation deferred to v2 deployment pipeline.")
print("\nG11 documented.")

In [ ]:
# ============================================================================
# Cell 17 — G12: Survival Analysis Framework (DIAGNOSTIC)
# ============================================================================
#
# Phase 2 Item 7: "Cox Proportional Hazards model required; binary UWL alert
# is operationally inadequate."
#
# Phase 3 scope: Kaplan-Meier estimation and Cox Proportional Hazards fitting
# are diagnostic exercises — they characterise the P-F interval distribution
# and quantify the hazard function, but do not produce features for the
# feature matrix.  The survival model outputs (hazard ratios, survival
# probabilities) are informational inputs for Phase 4 model interpretation
# and Phase 5 maintenance scheduling.
#
# Resolution: DOCUMENTED — survival analysis diagnostic completed here;
# integration into the scheduling decision layer deferred to Phase 5.
# ============================================================================

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

print("G12 — Survival Analysis Framework\n")
print("Phase 2 mandate: Cox PH model required; binary UWL alert operationally")
print("inadequate (P-F spread CV = 1.052, range 1–464 cycles).\n")

# ── Step 1: Compute P-F interval per engine ──────────────────────────────
# The P-F interval is defined as the number of cycles between UWL breach
# (|z_s4| > 2) and engine failure (final cycle).  Engines that never
# breach UWL are right-censored observations.

UWL_THRESHOLD = 2.0
UNIT_KEY_LOCAL = ["subset_origin", "unit_id"]

def compute_pf_intervals(df):
    """Compute P-F interval and censoring indicator per engine."""
    records = []
    for (subset, uid), grp in df.groupby(UNIT_KEY_LOCAL):
        grp_sorted = grp.sort_values("cycle")
        total_cycles = len(grp_sorted)

        # Identify first UWL breach in s4 (EGT).
        breach_mask = grp_sorted["s4"].abs() > UWL_THRESHOLD
        if breach_mask.any():
            first_breach_idx = breach_mask.idxmax()
            breach_cycle = grp_sorted.loc[first_breach_idx, "cycle"]
            final_cycle  = grp_sorted["cycle"].max()
            pf_interval  = final_cycle - breach_cycle
            censored     = 0  # Event observed.
        else:
            # Engine never breached UWL — right-censored at total life.
            pf_interval = total_cycles
            censored    = 1  # Right-censored.

        # Fault mode family assignment.
        fault_family = 1 if subset in ("FD003", "FD004") else 0
        fault_label  = "Dual-fault" if fault_family == 1 else "Single-fault"

        records.append({
            "subset_origin":    subset,
            "unit_id":          uid,
            "pf_interval":      pf_interval,
            "censored":         censored,
            "event_observed":   1 - censored,
            "fault_mode_family": fault_family,
            "fault_label":      fault_label,
            "total_life":       total_cycles,
        })
    return pd.DataFrame(records)

pf_df = compute_pf_intervals(fd00u_train)

n_total    = len(pf_df)
n_events   = pf_df["event_observed"].sum()
n_censored = pf_df["censored"].sum()

print(f"  Engines analysed:    {n_total}")
print(f"  Events observed:     {n_events} (UWL breach → failure)")
print(f"  Right-censored:      {n_censored} (never breached UWL)")
print(f"  Censoring fraction:  {n_censored / n_total:.1%}")

# ── Step 2: Kaplan-Meier survival curves ─────────────────────────────────
# Overall and stratified by fault mode family.

kmf_overall = KaplanMeierFitter()
kmf_overall.fit(
    durations=pf_df["pf_interval"],
    event_observed=pf_df["event_observed"],
    label="All Engines"
)

kmf_single = KaplanMeierFitter()
kmf_dual   = KaplanMeierFitter()

single_mask = pf_df["fault_mode_family"] == 0
dual_mask   = pf_df["fault_mode_family"] == 1

kmf_single.fit(
    durations=pf_df.loc[single_mask, "pf_interval"],
    event_observed=pf_df.loc[single_mask, "event_observed"],
    label="Single-fault (FD001/FD002)"
)
kmf_dual.fit(
    durations=pf_df.loc[dual_mask, "pf_interval"],
    event_observed=pf_df.loc[dual_mask, "event_observed"],
    label="Dual-fault (FD003/FD004)"
)

# Log-rank test for statistical significance of family difference.
lr_result = logrank_test(
    pf_df.loc[single_mask, "pf_interval"],
    pf_df.loc[dual_mask, "pf_interval"],
    event_observed_A=pf_df.loc[single_mask, "event_observed"],
    event_observed_B=pf_df.loc[dual_mask, "event_observed"],
)

print(f"\n  Log-rank test (single vs dual fault):")
print(f"    Test statistic: {lr_result.test_statistic:.3f}")
print(f"    p-value:        {lr_result.p_value:.4e}")
sig_label = "SIGNIFICANT" if lr_result.p_value < 0.05 else "NOT SIGNIFICANT"
print(f"    Result:         {sig_label} at alpha = 0.05")

# ── Visualisation 1: Kaplan-Meier Survival Curves ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Overall survival curve.
ax = axes[0]
kmf_overall.plot_survival_function(ax=ax, color=FC_DARK_BLUE, linewidth=2)
ax.axhline(y=0.5, color=FC_ORANGE, linestyle="--", linewidth=1, alpha=0.7,
           label="Median survival")
median_surv = kmf_overall.median_survival_time_
if pd.notna(median_surv):
    ax.axvline(x=median_surv, color=FC_ORANGE, linestyle=":", linewidth=1, alpha=0.5)
ax.set_title("Kaplan-Meier Survival Curve — All Engines")
ax.set_xlabel("P-F Interval (cycles)")
ax.set_ylabel("Survival Probability")
ax.legend(loc="upper right")
ax.grid(True)

# Panel B: Stratified by fault mode family.
ax = axes[1]
kmf_single.plot_survival_function(ax=ax, color=FC_DARK_BLUE, linewidth=2)
kmf_dual.plot_survival_function(ax=ax, color=FC_DEEP_RED, linewidth=2)
ax.axhline(y=0.5, color=FC_ORANGE, linestyle="--", linewidth=1, alpha=0.7)
ax.set_title("Kaplan-Meier Survival — Stratified by Fault Mode Family")
ax.set_xlabel("P-F Interval (cycles)")
ax.set_ylabel("Survival Probability")
ax.annotate(
    f"Log-rank p = {lr_result.p_value:.2e}",
    xy=(0.95, 0.95), xycoords="axes fraction",
    ha="right", va="top",
    fontsize=9, fontstyle="italic",
    bbox=dict(boxstyle="round,pad=0.3", facecolor=FC_LIGHT_GREY, alpha=0.8)
)
ax.legend(loc="upper right")
ax.grid(True)

plt.suptitle("G12 — Survival Analysis: P-F Interval Characterisation",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Step 3: Cox Proportional Hazards Model ───────────────────────────────
# Covariates: fault_mode_family + mean sensor z-scores at mid-life.

# Compute engine-level summary statistics for Cox covariates.
engine_stats = fd00u_train.groupby(UNIT_KEY_LOCAL).agg(
    s4_mean=("s4", "mean"),
    s7_mean=("s7", "mean"),
    s9_mean=("s9", "mean"),
    health_index_mean=("health_index", "mean"),
).reset_index()

cox_df = pf_df.merge(engine_stats, on=UNIT_KEY_LOCAL, how="left")

# Fit Cox PH model.
cox_features = ["fault_mode_family", "s4_mean", "s7_mean", "s9_mean"]
cox_input = cox_df[["pf_interval", "event_observed"] + cox_features].copy()

cph = CoxPHFitter()
cph.fit(cox_input, duration_col="pf_interval", event_col="event_observed")

print("\n  Cox Proportional Hazards Model — Summary")
print("  " + "─" * 60)
cph.print_summary(columns=["coef", "exp(coef)", "se(coef)", "p", "lower 0.95", "upper 0.95"])

# Concordance index.
c_index = cph.concordance_index_
print(f"\n  Concordance index (C-index): {c_index:.4f}")
print(f"  Interpretation: {'Good' if c_index > 0.65 else 'Moderate' if c_index > 0.55 else 'Weak'} "
      f"discriminative ability")

# ── Visualisation 2: Cox PH Hazard Ratios ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

summary = cph.summary
coefs   = summary["coef"]
ci_low  = summary["coef lower 95%"]
ci_high = summary["coef upper 95%"]
y_pos   = np.arange(len(coefs))

colours = [FC_DEEP_RED if p < 0.05 else FC_STEEL
           for p in summary["p"]]

ax.barh(y_pos, coefs, color=colours, edgecolor=FC_CHARCOAL, linewidth=0.5,
        height=0.6, alpha=0.85)
ax.errorbar(coefs, y_pos, xerr=[coefs - ci_low, ci_high - coefs],
            fmt="none", ecolor=FC_CHARCOAL, capsize=4, linewidth=1.2)
ax.axvline(x=0, color=FC_CHARCOAL, linestyle="-", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(coefs.index)
ax.set_xlabel("Cox PH Coefficient (log hazard ratio)")
ax.set_title("G12 — Cox Proportional Hazards: Covariate Effects on Failure Hazard")
ax.grid(True, axis="x")

# Legend for significance.
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=FC_DEEP_RED, edgecolor=FC_CHARCOAL, label="p < 0.05 (significant)"),
    Patch(facecolor=FC_STEEL, edgecolor=FC_CHARCOAL, label="p ≥ 0.05 (not significant)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

# ── Store diagnostic outputs for Phase 5 transfer ────────────────────────
g12_diagnostics = {
    "n_engines":         n_total,
    "n_events":          int(n_events),
    "n_censored":        int(n_censored),
    "median_survival":   float(median_surv) if pd.notna(median_surv) else None,
    "logrank_p":         float(lr_result.p_value),
    "cox_c_index":       float(c_index),
    "cox_coefficients":  cph.summary["coef"].to_dict(),
}

print("\nG12 resolved — survival analysis diagnostic completed.")
print("  Kaplan-Meier curves and Cox PH model fitted.")
print("  Integration into scheduling decision layer deferred to Phase 5.")

In [ ]:
# ============================================================================
# Cell 18 — G13: Per-Fault-Mode-Family UWL Recalibration (DIAGNOSTIC)
# ============================================================================
#
# Phase 2 Item 8: "CV disparity 0.869 to 1.419 across subsets requires
# family-specific thresholds."
#
# Phase 3 scope: Quantify the P-F interval distribution per fault mode family
# and per subset, demonstrating that a single fixed UWL threshold produces
# materially different lead-time distributions across fault mode families.
# Propose family-specific UWL thresholds calibrated to equalise the P10 lead
# time across families.
#
# Resolution: DOCUMENTED — family-specific thresholds computed and validated
# here; operational deployment of adaptive thresholds deferred to Phase 5
# maintenance scheduling.
# ============================================================================

print("G13 — Per-Fault-Mode-Family UWL Recalibration\n")
print("Phase 2 mandate: CV disparity 0.869 to 1.419 across subsets requires")
print("family-specific thresholds.\n")

# ── Step 1: P-F interval statistics by subset and fault family ───────────
# Reuse pf_df computed in G12.

print("  P-F Interval Statistics by Subset:")
print("  " + "─" * 65)

subset_stats = pf_df.groupby("subset_origin")["pf_interval"].agg(
    ["count", "mean", "std", "median"]
)
subset_stats["cv"] = subset_stats["std"] / subset_stats["mean"]
subset_stats["P10"] = pf_df.groupby("subset_origin")["pf_interval"].quantile(0.10)
subset_stats["P90"] = pf_df.groupby("subset_origin")["pf_interval"].quantile(0.90)

for subset in ["FD001", "FD002", "FD003", "FD004"]:
    if subset in subset_stats.index:
        row = subset_stats.loc[subset]
        print(f"    {subset}: n={int(row['count']):>3d}, "
              f"mean={row['mean']:.1f}, std={row['std']:.1f}, "
              f"CV={row['cv']:.3f}, P10={row['P10']:.0f}, P90={row['P90']:.0f}")

family_stats = pf_df.groupby("fault_label")["pf_interval"].agg(
    ["count", "mean", "std", "median"]
)
family_stats["cv"] = family_stats["std"] / family_stats["mean"]
family_stats["P10"] = pf_df.groupby("fault_label")["pf_interval"].quantile(0.10)
family_stats["P90"] = pf_df.groupby("fault_label")["pf_interval"].quantile(0.90)

print(f"\n  P-F Interval Statistics by Fault Mode Family:")
print("  " + "─" * 65)
for label in ["Single-fault", "Dual-fault"]:
    if label in family_stats.index:
        row = family_stats.loc[label]
        print(f"    {label}: n={int(row['count']):>3d}, "
              f"mean={row['mean']:.1f}, std={row['std']:.1f}, "
              f"CV={row['cv']:.3f}, P10={row['P10']:.0f}, P90={row['P90']:.0f}")

# ── Step 2: UWL threshold sweep ──────────────────────────────────────────
# Sweep UWL thresholds from 1.0 to 3.5 and compute the resulting P-F
# interval P10 for each fault mode family.  The goal is to find the
# family-specific threshold that equalises actionable lead time.

UWL_SWEEP = np.arange(1.0, 3.6, 0.25)

def compute_pf_for_threshold(df, threshold):
    """Compute P-F intervals for a given UWL threshold on s4."""
    records = []
    for (subset, uid), grp in df.groupby(UNIT_KEY_LOCAL):
        grp_sorted = grp.sort_values("cycle")
        breach_mask = grp_sorted["s4"].abs() > threshold
        if breach_mask.any():
            first_breach_idx = breach_mask.idxmax()
            breach_cycle = grp_sorted.loc[first_breach_idx, "cycle"]
            final_cycle  = grp_sorted["cycle"].max()
            pf_interval  = final_cycle - breach_cycle
            fault_family = 1 if subset in ("FD003", "FD004") else 0
            records.append({
                "pf_interval":       pf_interval,
                "fault_mode_family": fault_family,
            })
    return pd.DataFrame(records)

sweep_results = []
for thr in UWL_SWEEP:
    pf_thr = compute_pf_for_threshold(fd00u_train, thr)
    if len(pf_thr) == 0:
        continue
    for fm in [0, 1]:
        fm_data = pf_thr[pf_thr["fault_mode_family"] == fm]["pf_interval"]
        if len(fm_data) > 0:
            sweep_results.append({
                "threshold":    thr,
                "fault_family": "Single-fault" if fm == 0 else "Dual-fault",
                "n_engines":    len(fm_data),
                "mean_pf":      fm_data.mean(),
                "P10_pf":       fm_data.quantile(0.10),
                "P50_pf":       fm_data.median(),
                "cv_pf":        fm_data.std() / fm_data.mean() if fm_data.mean() > 0 else np.nan,
            })

sweep_df = pd.DataFrame(sweep_results)

# ── Visualisation 1: P-F Distribution by Fault Mode Family ──────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: P-F interval distribution (KDE) by fault family.
ax = axes[0]
for label, colour in [("Single-fault", FC_DARK_BLUE), ("Dual-fault", FC_DEEP_RED)]:
    data = pf_df[pf_df["fault_label"] == label]["pf_interval"]
    data.plot.kde(ax=ax, color=colour, linewidth=2, label=label, bw_method=0.3)
ax.set_title("P-F Interval Distribution by Fault Mode Family")
ax.set_xlabel("P-F Interval (cycles)")
ax.set_ylabel("Density")
ax.set_xlim(0, None)
ax.legend(loc="upper right")
ax.grid(True)

# Annotate CV values.
for i, label in enumerate(["Single-fault", "Dual-fault"]):
    if label in family_stats.index:
        cv_val = family_stats.loc[label, "cv"]
        ax.annotate(f"CV = {cv_val:.3f}", xy=(0.95, 0.85 - i * 0.10),
                    xycoords="axes fraction", ha="right",
                    fontsize=9, fontstyle="italic",
                    color=FC_DARK_BLUE if i == 0 else FC_DEEP_RED)

# Panel B: P-F interval boxplot by subset.
ax = axes[1]
subset_order = ["FD001", "FD002", "FD003", "FD004"]
available_subsets = [s for s in subset_order if s in pf_df["subset_origin"].unique()]
box_data = [pf_df[pf_df["subset_origin"] == s]["pf_interval"].values
            for s in available_subsets]

bp = ax.boxplot(box_data, tick_labels=available_subsets, patch_artist=True,
                medianprops=dict(color=FC_DEEP_RED, linewidth=2),
                whiskerprops=dict(color=FC_CHARCOAL),
                capprops=dict(color=FC_CHARCOAL))

box_colours = [FC_DARK_BLUE, FC_NAVY, FC_DEEP_RED, FC_STEEL]
for patch, colour in zip(bp["boxes"], box_colours[:len(bp["boxes"])]):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.set_title("P-F Interval Distribution by C-MAPSS Subset")
ax.set_xlabel("Subset")
ax.set_ylabel("P-F Interval (cycles)")
ax.grid(True, axis="y")

plt.suptitle("G13 — P-F Interval Variability Across Fault Mode Families",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Visualisation 2: UWL Threshold Sweep ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: P10 lead time vs threshold by family.
ax = axes[0]
for label, colour in [("Single-fault", FC_DARK_BLUE), ("Dual-fault", FC_DEEP_RED)]:
    fam_sweep = sweep_df[sweep_df["fault_family"] == label]
    ax.plot(fam_sweep["threshold"], fam_sweep["P10_pf"],
            color=colour, linewidth=2, marker="o", markersize=5, label=label)

# Reference line at 20 cycles (MRO minimum response time).
ax.axhline(y=20, color=FC_ORANGE, linestyle="--", linewidth=1.5,
           label="MRO response minimum (20 cycles)")
ax.set_title("P10 Lead Time vs UWL Threshold — By Fault Family")
ax.set_xlabel("UWL Threshold (|z|)")
ax.set_ylabel("P10 P-F Interval (cycles)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True)

# Panel B: CV vs threshold by family.
ax = axes[1]
for label, colour in [("Single-fault", FC_DARK_BLUE), ("Dual-fault", FC_DEEP_RED)]:
    fam_sweep = sweep_df[sweep_df["fault_family"] == label]
    ax.plot(fam_sweep["threshold"], fam_sweep["cv_pf"],
            color=colour, linewidth=2, marker="s", markersize=5, label=label)

ax.set_title("P-F Interval CV vs UWL Threshold — By Fault Family")
ax.set_xlabel("UWL Threshold (|z|)")
ax.set_ylabel("Coefficient of Variation")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True)

plt.suptitle("G13 — UWL Threshold Sensitivity: Family-Specific Calibration",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Step 3: Recommend family-specific thresholds ─────────────────────────
# Find the threshold per family that yields P10 >= 20 cycles (MRO minimum).

MRO_MIN_LEAD_TIME = 20

print("\n  Family-Specific UWL Threshold Recommendations:")
print("  " + "─" * 65)

recommended_thresholds = {}
for label in ["Single-fault", "Dual-fault"]:
    fam_sweep = sweep_df[sweep_df["fault_family"] == label].copy()
    # Find the tightest threshold (highest |z|) that still yields P10 >= MRO minimum.
    viable = fam_sweep[fam_sweep["P10_pf"] >= MRO_MIN_LEAD_TIME]
    if len(viable) > 0:
        best = viable.loc[viable["threshold"].idxmax()]
        recommended_thresholds[label] = best["threshold"]
        print(f"    {label}: UWL = |z| > {best['threshold']:.2f}  "
              f"(P10 = {best['P10_pf']:.0f} cycles, CV = {best['cv_pf']:.3f})")
    else:
        # All thresholds yield P10 < MRO minimum — use lowest threshold.
        best = fam_sweep.loc[fam_sweep["threshold"].idxmin()]
        recommended_thresholds[label] = best["threshold"]
        print(f"    {label}: UWL = |z| > {best['threshold']:.2f}  "
              f"(P10 = {best['P10_pf']:.0f} cycles — below MRO minimum)")

print(f"\n  Baseline (Phase 2): UWL = |z| > 2.00 (fixed, family-agnostic)")
print(f"  Recommended: family-specific thresholds as above.")
print(f"  Note: Adaptive thresholds operationalised in Phase 5 scheduling.")

# ── Store diagnostic outputs for Phase 5 transfer ────────────────────────
g13_diagnostics = {
    "subset_cv": subset_stats["cv"].to_dict() if "cv" in subset_stats.columns else {},
    "family_cv": family_stats["cv"].to_dict() if "cv" in family_stats.columns else {},
    "recommended_thresholds": recommended_thresholds,
    "mro_min_lead_time": MRO_MIN_LEAD_TIME,
}

print("\nG13 resolved — per-fault-mode-family UWL recalibration completed.")
print("  Family-specific thresholds calibrated to MRO minimum lead time.")
print("  Operational deployment deferred to Phase 5 scheduling layer.")

In [ ]:
# ============================================================================
# Cell 19 — Part A Summary: Gap Resolution Audit
# ============================================================================

print("=" * 70)
print("PART A — GAP RESOLUTION AUDIT SUMMARY")
print("=" * 70)

gap_audit = {
    "G1":  ("Composite key enforcement",            "RESOLVED"),
    "G2":  ("Temporal ordering",                     "RESOLVED"),
    "G3":  ("Dataset class imbalance",               "DOCUMENTED — deferred to Phase 4"),
    "G4":  ("Regime row imbalance",                  "DOCUMENTED — deferred to Phase 4"),
    "G5":  ("regime_id one-hot encoding",            "RESOLVED"),
    "G6":  ("s7 differential — fault_mode_family",   "RESOLVED"),
    "G7":  ("Linked pairs — VIF + ratio features",   "RESOLVED"),
    "G8":  ("Extreme z-scores — cycle-position",     "RESOLVED — retain all"),
    "G9":  ("s16 partial-dead — ablation deferred",  "DOCUMENTED — retained"),
    "G10": ("Health index (s4 + s7 + s9)",           "RESOLVED"),
    "G11": ("Regime OOD guard",                      "DOCUMENTED — deferred to v1"),
    "G12": ("Survival analysis framework",           "DOCUMENTED — deferred to Phase 5"),
    "G13": ("Per-family UWL recalibration",          "DOCUMENTED — deferred to Phase 5"),
}

for gap_id, (desc, status) in gap_audit.items():
    icon = "+" if "RESOLVED" in status or "DOCUMENTED" in status else "x"
    print(f"  [{icon}] {gap_id}: {desc}")
    print(f"           {status}")

print("=" * 70)
print("All 13 gaps addressed. Pipeline cleared for Part B (Feature Construction).")
print("=" * 70)

---

## Part B — Feature Construction (Dynamic Feature Vector)

The feature space is constructed in four explicit steps.
The total feature count is derived dynamically at runtime.

### Before/After Strategy

A snapshot of the DataFrame shape and columns is taken before feature engineering
begins and compared after completion to verify the exact number of features added.

In [ ]:
# ============================================================================
# Cell 21 — Before Snapshot & Active Sensor Variance Audit
# ============================================================================

# Capture before-state for Part B before/after comparison.
BEFORE_SHAPE_TRAIN = fd00u_train.shape
BEFORE_SHAPE_VAL   = fd00u_val.shape
BEFORE_COLS        = list(fd00u_train.columns)

print(f"BEFORE feature engineering:")
print(f"  Train shape: {BEFORE_SHAPE_TRAIN}")
print(f"  Val shape:   {BEFORE_SHAPE_VAL}")
print(f"  Columns ({len(BEFORE_COLS)}): {BEFORE_COLS}")

# Active sensor audit — derive N_ACTIVE from variance (not hardcoded).
print("\n--- Active Sensor Variance Audit ---\n")
sensor_variance = fd00u_train[SENSOR_COLS].var()
active_mask = sensor_variance > VARIANCE_THRESHOLD
N_ACTIVE = int(active_mask.sum())

print(f"  Variance threshold (tau): {VARIANCE_THRESHOLD}")
print(f"  Active sensors: {N_ACTIVE} / {len(SENSOR_COLS)}")
print()

dead_sensors = []
for col in SENSOR_COLS:
    status = "ACTIVE" if active_mask[col] else "DEAD"
    print(f"  {col:>4} ({SENSOR_NAMES.get(col, ''):>30}): "
          f"var = {sensor_variance[col]:.6f} -> {status}")
    if not active_mask[col]:
        dead_sensors.append(col)

active_sensor_columns = list(sensor_variance[active_mask].index)

# Cross-reference with Phase 2 findings.
# Phase 2 Technical Report confirmed:
#   - 15 sensors ACTIVE with full variance
#   - 4 sensors REGIME-DEAD: s1 (T2), s5 (P2), s18 (Nf_dmd), s19 (PCNfR_dmd)
#   - 2 sensors PARTIALLY DEAD / reduced-sigma: s6 (P15), s16 (farB)
# s6 and s16 pass the variance threshold (sigma > 0), so N_ACTIVE = 17.
EXPECTED_DEAD = ["s1", "s5", "s18", "s19"]

print(f"\n--- Dead Sensor Cross-Reference ---")
print(f"  Phase 2 regime-dead sensors: {EXPECTED_DEAD}")
print(f"  Empirical dead sensors:      {dead_sensors}")

dead_match = set(dead_sensors) == set(EXPECTED_DEAD)
if dead_match:
    print(f"  Cross-reference: MATCH — dead sensors are consistent with Phase 2 findings.")
else:
    print(f"  Cross-reference: MISMATCH — investigate discrepancy.")
assert dead_match, (
    f"Dead sensor set does not match Phase 2 findings. "
    f"Expected {EXPECTED_DEAD}, found {dead_sensors}."
)

# Store formula components for post-hoc verification in Cell 24.
N_BASE     = len(OP_COLS) + len(SENSOR_COLS)  # 3 + 21 = 24
N_VIRTUAL  = 3  # CPR, E_thermal, EGT Drift
N_FATIGUE  = 3  # Cumulative stress for s4, s9, s7

# Gap resolution columns added by Part A (Cells 10-15).
# These are in BEFORE_COLS but not in the original FD00u load.
ORIGINAL_BASE = set(["unit_id", "cycle"] + OP_COLS + SENSOR_COLS
                     + ["RUL", "subset", "subset_origin"])
GAP_RESOLUTION_COLS = sorted([c for c in BEFORE_COLS if c not in ORIGINAL_BASE])
N_GAP = len(GAP_RESOLUTION_COLS)

print(f"\n--- Formula Components (to be verified after engineering) ---")
print(f"  N_base     = {N_BASE} (3 ops + 21 sensors — all retained)")
print(f"  N_active   = {N_ACTIVE} (kinematic expansion applied to active sensors only)")
print(f"  N_virtual  = {N_VIRTUAL}")
print(f"  N_fatigue  = {N_FATIGUE}")
print(f"  N_gap      = {N_GAP} (gap-resolution features from Part A)")
for c in GAP_RESOLUTION_COLS:
    print(f"               - {c}")
print(f"  N_predicted = {N_BASE} + ({N_ACTIVE} x 3) + {N_VIRTUAL} + {N_FATIGUE} + {N_GAP}"
      f" = {N_BASE + (N_ACTIVE * 3) + N_VIRTUAL + N_FATIGUE + N_GAP}")
print(f"\n  These counts will be verified empirically after all engineering steps.")

In [ ]:
# ============================================================================
# Cell 22 — Step 1: Core Manifold Verification (24 Base Features)
# ============================================================================

print("Step 1 — Core Manifold (24 Features)\n")

core_features = OP_COLS + SENSOR_COLS
missing = [c for c in core_features if c not in fd00u_train.columns]
assert len(missing) == 0, f"Missing core features: {missing}"

print(f"  Operational settings: {OP_COLS}")
print(f"  Sensors: {SENSOR_COLS}")
print(f"  Core manifold size: {len(core_features)} features")
print(f"\nStep 1 verified — all 24 core features present.")

In [ ]:
# ============================================================================
# Cell 23 — Step 2: Kinematic Expansion (N_ACTIVE x 3 Features)
# ============================================================================

print(f"Step 2 — Kinematic Expansion ({N_ACTIVE} x 3 = {N_ACTIVE * 3} Features)\n")

# For each active sensor, generate three kinematic proxies:
#   1. Kinematic Velocity (first-order diff)
#   2. Thermodynamic Baseline (rolling mean, window=5)
#   3. Systemic Instability (rolling std, window=5)
#
# All operations are per-engine using composite key to prevent cross-engine
# contamination.

def kinematic_expansion(df, name):
    """Apply kinematic expansion per engine trajectory."""
    n_new = 0
    for col in active_sensor_columns:
        # 1. Kinematic Velocity.
        df[f"{col}_delta"] = df.groupby(UNIT_KEY)[col].diff()

        # 2. Thermodynamic Baseline (rolling mean).
        df[f"{col}_rmean"] = df.groupby(UNIT_KEY)[col].transform(
            lambda x: x.rolling(window=ROLLING_WINDOW, min_periods=1).mean()
        )

        # 3. Systemic Instability (rolling std).
        df[f"{col}_rstd"] = df.groupby(UNIT_KEY)[col].transform(
            lambda x: x.rolling(window=ROLLING_WINDOW, min_periods=1).std()
        )
        n_new += 3

    # Fill NaN from rolling/diff operations with zero.
    # Physical justification: Velocity=0 (no change), RMean=0 (no baseline),
    # RStd=0 (no instability), Fatigue starts at 0.
    df.fillna(0, inplace=True)

    print(f"  [{name}] {n_new} kinematic features added ({N_ACTIVE} sensors x 3)")
    return df

fd00u_train = kinematic_expansion(fd00u_train, "Train")
fd00u_val   = kinematic_expansion(fd00u_val, "Val")

# Verify.
delta_cols = [c for c in fd00u_train.columns if c.endswith("_delta")]
rmean_cols = [c for c in fd00u_train.columns if c.endswith("_rmean")]
rstd_cols  = [c for c in fd00u_train.columns if c.endswith("_rstd")]

print(f"\n  Delta columns:  {len(delta_cols)}")
print(f"  RMean columns:  {len(rmean_cols)}")
print(f"  RStd columns:   {len(rstd_cols)}")
print(f"  Total kinematic: {len(delta_cols) + len(rmean_cols) + len(rstd_cols)}")
print(f"\nStep 2 complete — {N_ACTIVE * 3} kinematic features added.")

In [ ]:
# ============================================================================
# Cell 24 — Step 3: Thermodynamic Virtual Sensors (3 Features)
# ============================================================================

print("Step 3 — Thermodynamic Virtual Sensors (3 Features)\n")

# Note: Virtual sensor formulae reference physical sensor ratios.
# In z-normalised space, denominators can be near-zero.
# Safe division guards against numerical explosion.
SAFE_DENOM_THRESHOLD = 1e-6

# 1. Compressor Pressure-Temperature Ratio (CPR) = P30 / T30 = s7 / s3
# Original spec: s7/s5 (P30/P2).  s5 is regime-dead in z-space (var ≈ 0),
# making that ratio degenerate (all zeros after safe division).
# Replacement: s7/s3 — total pressure to temperature at HPC outlet,
# proportional to gas density (ideal gas: rho ∝ P/T).  Both sensors are
# live with non-trivial variance; the ratio captures compressor health.
for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
    denom = df_loop[CPR_DENOMINATOR]
    safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD, np.nan)
    df_loop["CPR"] = (df_loop[CPR_NUMERATOR] / safe_denom).fillna(0)

print(f"  1. CPR = {CPR_NUMERATOR} / {CPR_DENOMINATOR}  "
      f"[replaced s7/s5 → s7/s3; s5 regime-dead]")
print(f"     Train — mean: {fd00u_train['CPR'].mean():.4f}, "
      f"std: {fd00u_train['CPR'].std():.4f}")

# 2. Thermal Efficiency Proxy = (phi x Ps30) / Nc = (s12 x s11) / s9
for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
    numerator = df_loop[E_THERMAL_NUM_A] * df_loop[E_THERMAL_NUM_B]
    denom = df_loop[E_THERMAL_DENOM]
    safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD, np.nan)
    df_loop["E_thermal"] = (numerator / safe_denom).fillna(0)

print(f"  2. E_thermal = ({E_THERMAL_NUM_A} x {E_THERMAL_NUM_B}) / {E_THERMAL_DENOM}")
print(f"     Train — mean: {fd00u_train['E_thermal'].mean():.4f}, "
      f"std: {fd00u_train['E_thermal'].std():.4f}")

# 3. EGT Margin Drift = T50 - mu_baseline = s4 - mu(s4 for Internal Train)
# mu_baseline is computed from Internal Train only (zero-leakage).
egt_baseline = fd00u_train[EGT_SENSOR].mean()

for df_loop, name in [(fd00u_train, "Train"), (fd00u_val, "Val")]:
    df_loop["EGT_drift"] = df_loop[EGT_SENSOR] - egt_baseline

print(f"  3. EGT_drift = {EGT_SENSOR} - mu_baseline (mu = {egt_baseline:.4f})")
print(f"     Train — mean: {fd00u_train['EGT_drift'].mean():.4f}, "
      f"std: {fd00u_train['EGT_drift'].std():.4f}")

print(f"\nStep 3 complete — 3 virtual sensors added (CPR, E_thermal, EGT_drift).")

In [ ]:
# ============================================================================
# Cell 25 — Step 4: Cumulative Fatigue Index — Miner's Rule Proxy (3 Features)
# ============================================================================

print("Step 4 — Cumulative Fatigue Index (3 Features)\n")

# D(t) = sum_{i=1}^{t} max(0, z_i)
# Only positive z-score deviations summed — damage accumulates only when
# running hotter or harder than the healthy regime baseline.

def cumulative_fatigue(df, name):
    """Compute cumulative fatigue index per engine for each fatigue sensor."""
    # Defensive re-sort to guarantee monotonic cycle order within each engine.
    # cumsum is order-dependent — row ordering must be deterministic.
    df = df.sort_values(UNIT_KEY + ["cycle"]).reset_index(drop=True)

    for sensor in FATIGUE_SENSORS:
        fatigue_col = f"{sensor}_cumfatigue"
        positive_z = df[sensor].clip(lower=0)
        df[fatigue_col] = positive_z.groupby(
            [df["subset_origin"], df["unit_id"]]
        ).cumsum()
        term_mean = df.loc[df["RUL"] < 10, fatigue_col].mean()
        print(f"  [{name}] {fatigue_col}: max = {df[fatigue_col].max():.2f}, "
              f"mean(terminal) = {term_mean:.2f}")
    return df

fd00u_train = cumulative_fatigue(fd00u_train, "Train")
fd00u_val   = cumulative_fatigue(fd00u_val, "Val")

# Verify monotonicity for a sample engine.
sample_engine = fd00u_train.groupby(UNIT_KEY).size().idxmax()
sample_mask = (
    (fd00u_train["subset_origin"] == sample_engine[0]) &
    (fd00u_train["unit_id"] == sample_engine[1])
)
sample_df = fd00u_train[sample_mask]

fig, ax = plt.subplots(figsize=(12, 4))
for sensor in FATIGUE_SENSORS:
    fatigue_col = f"{sensor}_cumfatigue"
    ax.plot(sample_df["cycle"], sample_df[fatigue_col],
            linewidth=1.5, label=f"{sensor} ({SENSOR_NAMES.get(sensor, '')})")

ax.set_title(f"Cumulative Fatigue Index — Engine ({sample_engine[0]}, "
             f"unit {sample_engine[1]})")
ax.set_xlabel("Cycle")
ax.set_ylabel("Cumulative Fatigue D(t)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

# Verify strict non-decrease (with float tolerance).
MONO_TOL = -1e-10
for sensor in FATIGUE_SENSORS:
    fatigue_col = f"{sensor}_cumfatigue"
    diffs = sample_df[fatigue_col].diff().dropna()
    assert (diffs >= MONO_TOL).all(), f"{fatigue_col} is not monotonically non-decreasing!"

print(f"\nStep 4 complete — 3 cumulative fatigue indices added.")
print(f"  All curves verified strictly non-decreasing (Miner's Rule satisfied).")

In [ ]:
# ============================================================================
# Cell 26 — After Snapshot: Empirical Verification & Descriptive Statistics
# ============================================================================

AFTER_SHAPE_TRAIN = fd00u_train.shape
AFTER_SHAPE_VAL   = fd00u_val.shape
AFTER_COLS        = list(fd00u_train.columns)

# Columns added by Steps 2-4 (Part B).
new_cols = [c for c in AFTER_COLS if c not in BEFORE_COLS]

# Gap resolution columns were added in Part A (before the BEFORE snapshot),
# so they don't appear in new_cols.  Use the explicit list from Cell 19.
gap_cols = GAP_RESOLUTION_COLS

print("=" * 70)
print("BEFORE / AFTER FEATURE ENGINEERING COMPARISON")
print("=" * 70)
print(f"\n  BEFORE (after Part A gap resolution):")
print(f"    Train: {BEFORE_SHAPE_TRAIN}")
print(f"    Val:   {BEFORE_SHAPE_VAL}")
print(f"    Columns: {len(BEFORE_COLS)}")
print(f"\n  AFTER (after Part B feature engineering):")
print(f"    Train: {AFTER_SHAPE_TRAIN}")
print(f"    Val:   {AFTER_SHAPE_VAL}")
print(f"    Columns: {len(AFTER_COLS)}")

# --- Empirical categorisation ---
kinematic_cols = [c for c in new_cols if c.endswith(("_delta", "_rmean", "_rstd"))]
virtual_cols   = [c for c in new_cols if c in ["CPR", "E_thermal", "EGT_drift"]]
fatigue_cols   = [c for c in new_cols if c.endswith("_cumfatigue")]
uncategorised  = [c for c in new_cols if c not in kinematic_cols
                  and c not in virtual_cols and c not in fatigue_cols]

print(f"\n  Columns added by Part A (Gap Resolution): {len(gap_cols)}")
for c in gap_cols:
    print(f"    - {c}")
print(f"  Columns added by Part B (Feature Engineering): {len(new_cols)}")
print(f"    Kinematic (Step 2): {len(kinematic_cols)}")
print(f"    Virtual (Step 3):   {len(virtual_cols)}")
for c in virtual_cols:
    print(f"      - {c}")
print(f"    Fatigue (Step 4):   {len(fatigue_cols)}")
for c in fatigue_cols:
    print(f"      - {c}")
if uncategorised:
    print(f"    Uncategorised:      {len(uncategorised)}")
    for c in uncategorised:
        print(f"      - {c}")

# --- Empirical verification against formula ---
print("\n" + "=" * 70)
print("EMPIRICAL VERIFICATION — Formula vs Actual")
print("=" * 70)

n_kinematic_expected = N_ACTIVE * 3
n_kinematic_actual   = len(kinematic_cols)
n_virtual_expected   = N_VIRTUAL
n_virtual_actual     = len(virtual_cols)
n_fatigue_expected   = N_FATIGUE
n_fatigue_actual     = len(fatigue_cols)
n_gap_expected       = N_GAP
n_gap_actual         = len(gap_cols)

N_TOTAL_EXPECTED = (N_BASE + n_kinematic_expected + n_virtual_expected
                    + n_fatigue_expected + n_gap_expected)

def _match(exp, act):
    return "MATCH" if exp == act else "MISMATCH"

print(f"  Gap resol.: expected {n_gap_expected}, actual {n_gap_actual}  {_match(n_gap_expected, n_gap_actual)}")
print(f"  Kinematic:  expected {n_kinematic_expected} ({N_ACTIVE} x 3), actual {n_kinematic_actual}  {_match(n_kinematic_expected, n_kinematic_actual)}")
print(f"  Virtual:    expected {n_virtual_expected}, actual {n_virtual_actual}  {_match(n_virtual_expected, n_virtual_actual)}")
print(f"  Fatigue:    expected {n_fatigue_expected}, actual {n_fatigue_actual}  {_match(n_fatigue_expected, n_fatigue_actual)}")
print(f"\n  Formula total (N_base + kinematic + virtual + fatigue + gap):")
print(f"    {N_BASE} + {n_kinematic_expected} + {n_virtual_expected} "
      f"+ {n_fatigue_expected} + {n_gap_expected} = {N_TOTAL_EXPECTED}")

assert n_kinematic_expected == n_kinematic_actual, (
    f"Kinematic mismatch: expected {n_kinematic_expected}, actual {n_kinematic_actual}"
)
assert n_virtual_expected == n_virtual_actual, (
    f"Virtual mismatch: expected {n_virtual_expected}, actual {n_virtual_actual}"
)
assert n_fatigue_expected == n_fatigue_actual, (
    f"Fatigue mismatch: expected {n_fatigue_expected}, actual {n_fatigue_actual}"
)
assert n_gap_expected == n_gap_actual, (
    f"Gap mismatch: expected {n_gap_expected}, actual {n_gap_actual}"
)
print("\n  All component counts VERIFIED against formula.")

# --- Descriptive statistics: before vs after ---
print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS — Before vs After (Internal Train)")
print("=" * 70)

# Before: feature columns only (exclude metadata).
METADATA_FOR_STATS = ["unit_id", "cycle", "subset", "subset_origin", "RUL"]
before_numeric = [c for c in BEFORE_COLS if c not in METADATA_FOR_STATS
                  and fd00u_train[c].dtype in ['float64', 'float32', 'int64', 'int32']]
print(f"\n  BEFORE — feature columns ({len(before_numeric)} numeric):")
before_stats = fd00u_train[before_numeric].describe().T[["mean", "std", "min", "max"]]
print(before_stats.round(4).to_string())

# After: new columns (Steps 2-4).
after_numeric = [c for c in new_cols
                 if fd00u_train[c].dtype in ['float64', 'float32', 'int64', 'int32']]
if after_numeric:
    print(f"\n  AFTER — engineered features ({len(after_numeric)} numeric):")
    eng_stats = fd00u_train[after_numeric].describe().T[["mean", "std", "min", "max"]]
    print(eng_stats.round(4).to_string())

print("\n" + "=" * 70)

---

## Gate Verification

The following cells implement the six Phase 3 gate checks specified in
Phase 3 Gate — Feature Engineering.

In [ ]:
# ============================================================================
# Cell 28 — Construct Feature Matrix X and Target Vector y
# ============================================================================

print("Constructing feature matrix X and target vector y...\n")

# Metadata columns that must NOT enter the feature matrix.
# Critical Prohibition: cycle and unit_id must not appear in X.
# "subset" is a Phase 2 provenance column identical to subset_origin — must
# also be excluded to prevent string-column leakage into the feature matrix.
METADATA_COLS = ["unit_id", "cycle", "subset", "subset_origin", "RUL"]

feature_cols = [c for c in fd00u_train.columns if c not in METADATA_COLS]
feature_cols_sorted = sorted(feature_cols)

X_train = fd00u_train[feature_cols_sorted].copy()
y_train = fd00u_train["RUL"].copy()
X_val   = fd00u_val[feature_cols_sorted].copy()
y_val   = fd00u_val["RUL"].copy()

print(f"  Total DataFrame columns: {len(fd00u_train.columns)}")
print(f"  Metadata excluded ({len(METADATA_COLS)}): {METADATA_COLS}")
print(f"  Feature columns:  {len(feature_cols_sorted)}")
print()

print(f"Feature columns ({len(feature_cols_sorted)}):")
for i, col in enumerate(feature_cols_sorted):
    print(f"  {i+1:>3}. {col}")

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape:   {y_val.shape}")

# Verify no string columns leaked into X.
string_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
assert len(string_cols) == 0, f"String columns in feature matrix: {string_cols}"
print(f"\nNo string columns in feature matrix — dtype check passed.")

In [ ]:
# ============================================================================
# Cell 29 — Gate 1: Active Sensor Count
# ============================================================================

print("Gate 1 — Active Sensor Count\n")

# Phase 2 confirmed 4 regime-dead sensors (s1, s5, s18, s19) with zero
# within-regime variance. N_ACTIVE = 17 is the correct empirical result.
# The kinematic expansion is applied only to active sensors; all 21 sensors
# are retained in the core manifold per Critical Prohibition #3.

gate1_pass = (N_ACTIVE == 17) and (len(dead_sensors) == 4)

print(f"  N_ACTIVE      = {N_ACTIVE}")
print(f"  Dead sensors  = {dead_sensors}")
print(f"  Expected: N_ACTIVE = 17 (Phase 2 confirmed 4 regime-dead)")
print(f"  Result:  {'PASS' if gate1_pass else 'FAIL'}")

assert gate1_pass, (
    f"Gate 1 FAILED: N_ACTIVE = {N_ACTIVE}, expected 17 "
    f"(consistent with Phase 2 regime-dead classification)"
)

In [ ]:
# ============================================================================
# Cell 30 — Gate 2: Total Feature Count
# ============================================================================

print("Gate 2 — Total Feature Count\n")

# N_TOTAL_EXPECTED includes all components: base + kinematic + virtual +
# fatigue + gap resolution.
n_features_actual = X_train.shape[1]

print(f"  Formula breakdown:")
print(f"    N_base (ops + sensors):    {N_BASE}")
print(f"    N_kinematic ({N_ACTIVE} x 3):     {N_ACTIVE * 3}")
print(f"    N_virtual:                 {N_VIRTUAL}")
print(f"    N_fatigue:                 {N_FATIGUE}")
print(f"    N_gap (Part A resolution): {N_GAP}")
print(f"    ─────────────────────────────")
print(f"    N_TOTAL expected:          {N_TOTAL_EXPECTED}")
print(f"  Actual feature count:        {n_features_actual}")

gate2_pass = (n_features_actual == N_TOTAL_EXPECTED)
status = "PASS" if gate2_pass else "FAIL"
print(f"\n  Actual ({n_features_actual}) == Expected ({N_TOTAL_EXPECTED}): {status}")

assert gate2_pass, (
    f"Gate 2 FAILED: {n_features_actual} features, expected {N_TOTAL_EXPECTED}"
)

In [ ]:
# ============================================================================
# Cell 31 — Gate 3: Feature Matrix Shape
# ============================================================================

print("Gate 3 — Feature Matrix Shape\n")

rows_match_train = (X_train.shape[0] == fd00u_train.shape[0])
rows_match_val   = (X_val.shape[0] == fd00u_val.shape[0])
cols_match       = (X_train.shape[1] == X_val.shape[1])

gate3_pass = rows_match_train and rows_match_val and cols_match

print(f"  X_train rows ({X_train.shape[0]:,}) == FD00u_train ({fd00u_train.shape[0]:,}): "
      f"{'OK' if rows_match_train else 'FAIL'}")
print(f"  X_val rows ({X_val.shape[0]:,}) == FD00u_val ({fd00u_val.shape[0]:,}): "
      f"{'OK' if rows_match_val else 'FAIL'}")
print(f"  X_train cols ({X_train.shape[1]}) == X_val cols ({X_val.shape[1]}): "
      f"{'OK' if cols_match else 'FAIL'}")
print(f"\n  Result: {'PASS' if gate3_pass else 'FAIL'}")

assert gate3_pass, "Gate 3 FAILED: Shape mismatch"

In [ ]:
# ============================================================================
# Cell 32 — Gate 4: NaN Audit
# ============================================================================

print("Gate 4 — NaN Audit\n")

nan_train = X_train.isna().sum().sum()
nan_val   = X_val.isna().sum().sum()

gate4_pass = (nan_train == 0) and (nan_val == 0)

print(f"  X_train NaN count: {nan_train}")
print(f"  X_val NaN count:   {nan_val}")
print(f"\n  Result: {'PASS' if gate4_pass else 'FAIL'}")

if not gate4_pass:
    nan_cols_train = X_train.columns[X_train.isna().any()].tolist()
    nan_cols_val   = X_val.columns[X_val.isna().any()].tolist()
    print(f"  NaN columns (Train): {nan_cols_train}")
    print(f"  NaN columns (Val):   {nan_cols_val}")

assert gate4_pass, f"Gate 4 FAILED: {nan_train + nan_val} NaN values remaining"

In [ ]:
# ============================================================================
# Cell 33 — Gate 5: Virtual Sensor Distributions
# ============================================================================

print("Gate 5 — Virtual Sensor Distributions\n")

virtual_sensors = ["CPR", "E_thermal", "EGT_drift"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

gate5_pass = True
for i, vs in enumerate(virtual_sensors):
    data = X_train[vs]
    std = data.std()
    skew_val = data.skew()
    kurt_val = data.kurtosis()

    is_nondegen = std > 0
    if not is_nondegen:
        gate5_pass = False

    print(f"  {vs}:")
    print(f"    Mean = {data.mean():.4f}, Std = {std:.4f}")
    print(f"    Skew = {skew_val:.4f}, Kurtosis = {kurt_val:.4f}")
    print(f"    Non-degenerate: {'YES' if is_nondegen else 'NO'}")

    sns.kdeplot(data, ax=axes[i], color=FC_DEEP_RED, fill=True, alpha=0.3,
                linewidth=2, warn_singular=False)
    axes[i].set_title(f"{vs} Distribution")
    axes[i].set_xlabel(vs)
    axes[i].axvline(data.mean(), color=FC_ORANGE, linestyle="--",
                    label=f"mu={data.mean():.2f}")
    axes[i].legend()

plt.suptitle("Virtual Sensor Distributions (Internal Train)", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n  Result: {'PASS' if gate5_pass else 'FAIL'}")
assert gate5_pass, "Gate 5 FAILED: One or more virtual sensors are degenerate"

In [ ]:
# ============================================================================
# Cell 34 — Gate 6: Cumulative Fatigue Monotonicity Plot
# ============================================================================

print("Gate 6 — Cumulative Fatigue Monotonicity Verification\n")

# Float tolerance for monotonicity check — cumsum of clipped-positive values
# should be strictly non-decreasing, but IEEE 754 arithmetic may produce
# diffs of order -1e-15. Tolerance of -1e-10 catches genuine violations
# while accepting machine-precision artefacts.
MONO_TOL = -1e-10

gate6_pass = True
sample_engines = fd00u_train.groupby(UNIT_KEY).size().nlargest(5).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for k, sensor in enumerate(FATIGUE_SENSORS):
    fatigue_col = f"{sensor}_cumfatigue"
    for j, engine_key in enumerate(sample_engines):
        mask = (
            (fd00u_train["subset_origin"] == engine_key[0]) &
            (fd00u_train["unit_id"] == engine_key[1])
        )
        eng_df = fd00u_train[mask].sort_values("cycle")
        axes[k].plot(eng_df["cycle"], eng_df[fatigue_col], linewidth=1.0,
                     label=f"{engine_key[0]}-U{engine_key[1]}")

        diffs = eng_df[fatigue_col].diff().dropna()
        if not (diffs >= MONO_TOL).all():
            min_diff = diffs.min()
            gate6_pass = False
            print(f"  FAIL: {fatigue_col} NOT monotonic for engine {engine_key} "
                  f"(min diff = {min_diff:.2e})")

    axes[k].set_title(f"{sensor} ({SENSOR_NAMES.get(sensor, '')})")
    axes[k].set_xlabel("Cycle")
    axes[k].set_ylabel("D(t)")
    axes[k].legend(fontsize=7)

fig.suptitle("Cumulative Fatigue Index — Representative Engines", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n  Result: {'PASS' if gate6_pass else 'FAIL'}")
assert gate6_pass, "Gate 6 FAILED: Cumulative fatigue is not monotonically non-decreasing"

---

## Save Outputs & Summary

In [ ]:
# ============================================================================
# Cell 36 — Save Feature Matrix & Engineered FD00u to Google Drive
# ============================================================================

os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Save full engineered DataFrames (with metadata) for traceability.
fd00u_train.to_parquet(OUTPUTS_DIR / "fd00u_train_featured.parquet", index=False)
fd00u_val.to_parquet(OUTPUTS_DIR / "fd00u_val_featured.parquet", index=False)

# Save feature matrices (X) and target vectors (y) for direct Phase 4 consumption.
X_train.to_parquet(OUTPUTS_DIR / "X_train.parquet", index=False)
X_val.to_parquet(OUTPUTS_DIR / "X_val.parquet", index=False)
y_train.to_frame("RUL").to_parquet(OUTPUTS_DIR / "y_train.parquet", index=False)
y_val.to_frame("RUL").to_parquet(OUTPUTS_DIR / "y_val.parquet", index=False)

# Save feature column list for reproducibility.
feature_manifest = pd.DataFrame({"feature": feature_cols_sorted})
feature_manifest.to_csv(OUTPUTS_DIR / "feature_manifest.csv", index=False)

print("Outputs saved to Google Drive:")
print(f"  fd00u_train_featured.parquet  ({fd00u_train.shape})")
print(f"  fd00u_val_featured.parquet    ({fd00u_val.shape})")
print(f"  X_train.parquet               ({X_train.shape})")
print(f"  X_val.parquet                 ({X_val.shape})")
print(f"  y_train.parquet               ({y_train.shape})")
print(f"  y_val.parquet                 ({y_val.shape})")
print(f"  feature_manifest.csv          ({len(feature_cols_sorted)} features)")

In [ ]:
# ============================================================================
# Cell 37 — Phase 3 Gate Verification Summary
# ============================================================================

gates = {
    "Gate 1": ("Active sensor count = 17 (Phase 2 regime-dead cross-ref)", gate1_pass),
    "Gate 2": (f"Total feature count == {N_TOTAL_EXPECTED} (full formula)", gate2_pass),
    "Gate 3": ("Feature matrix shape consistent", gate3_pass),
    "Gate 4": ("Zero NaN values remaining", gate4_pass),
    "Gate 5": ("Virtual sensor distributions non-degenerate", gate5_pass),
    "Gate 6": ("Cumulative fatigue strictly non-decreasing", gate6_pass),
}

print("=" * 70)
print("PHASE 3 — GATE VERIFICATION SUMMARY")
print("=" * 70)

all_pass = True
for gate_name, (description, passed) in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {gate_name}: {description}")
    print(f"           Result: {status}")
    if not passed:
        all_pass = False

print("=" * 70)
if all_pass:
    print("ALL 6 GATES PASSED — Phase 3 complete.")
    print("The pipeline is cleared to proceed to Phase 4 (Forensic Stress Test).")
else:
    print("ONE OR MORE GATES FAILED — investigate before proceeding.")
print("=" * 70)

assert all_pass, "Phase 3 gate verification failed. Do not proceed to Phase 4."

---

**Phase 3 complete.** The engineered feature matrix is saved to Google Drive
and ready for Phase 4 (Forensic Stress Test — XGBoost baseline + SHAP audit +
Perturbation Analysis).

**Outputs produced:**
- `fd00u_train_featured.parquet` / `fd00u_val_featured.parquet` — full engineered DataFrames with metadata
- `X_train.parquet` / `X_val.parquet` — feature matrices (metadata excluded)
- `y_train.parquet` / `y_val.parquet` — target vectors
- `feature_manifest.csv` — ordered list of all feature column names

**Gap resolution:** All 13 gaps from Phase 1/2 reports addressed (7 RESOLVED, 6 DOCUMENTED with justified deferrals).

**Feature engineering:** 81+ features constructed via the four-step pipeline (Core Manifold, Kinematic Expansion, Virtual Sensors, Cumulative Fatigue), with additional gap-resolution features.